# HeatUp 03 - analysis (four independent checks)

Reads what is on disk and judges it. **No calculator, no calculation** - results computed by notebook 02, by DFT, or
by anything else that writes the same files all work (see the end of this notebook).

| check | question | evidence |
|---|---|---|
| **mechanical** | positive-definite C (Born-Huang)? | relaxed-ion elastic tensor |
| **structural transition** | did the structure stay in its basin? | block-averaged positions / cell / symmetry breaking from the MD |
| **vibrational** | soft or imaginary modes? | the sources of `cfg.vibrational.sources`: harmonic phonon bands, quasi-harmonic volumes, MD VDOS (`anharmonic`) |
| **thermodynamic** | above the T-dependent hull of its competitors? | G(T) of the selected `cfg.terms` with the *same* model for every phase |

The checks are **independent and unordered**: the ones asked for in notebook 01 are reported (override with `cfg.gates.enabled`), any one can be asked for alone, every one is
evaluated whatever the others say. Verdicts are always recomputed from cached results, so a threshold never needs a rerun. MD temperatures and
the operating temperature come from each phase's `plan.json` and from the MD runs that exist - nothing to type here.

In [ ]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, override, status, run_analysis, evaluate, evaluate_all
from heatup.analyze.report import (gate_table, style_status, show_status, status_table, show_stability_figure,
                                   plot_free_energy_terms, plot_vibrational, plot_perturbative, vibrational_table, print_layout)
from heatup.core import contributions

warnings.filterwarnings("ignore")
heatup.verbose()
if os.path.exists("heatup_project.json"):
    heatup.load_json("heatup_project.json")      # folder roots written by notebook 01
print("HeatUp", heatup.__version__, "|", cfg.phases.database_root)

## Configuration

In [ ]:
# -- how trajectories are read (re-runs only the cheap analysis steps) ----------------------------------
cfg.analysis.equil_ps = 5.0          # discarded before analysis (ps)
cfg.analysis.stride = 1
cfg.analysis.block_ps = 2.0          # first/last averaging window of the structural-transition check (ps)
cfg.analysis.vdos_max_lag_ps = 10.0  # longest VACF lag of the VDOS (ps)
cfg.analysis.mobile_species = []     # e.g. ["Li"] for a superionic conductor (excluded from the structural-transition and vibrational checks)

# -- which checks, and their thresholds (never trigger a recomputation) ----------------------------------
cfg.gates.enabled = None  # None = the checks asked for in notebook 01 (cfg.plan.gates); or e.g. ["mechanical", "thermodynamic"]
cfg.gates.hull_ok_eV = 0.010
cfg.gates.hull_fail_eV = 0.100

# -- free energy and hull ---------------------------------------------------------------------------------
# WHICH terms enter G(T) is cfg.terms (set in notebook 01 and stored in every plan.json; None = as prepared).  Override here only
# on purpose, e.g. cfg.terms.vibrational = "harmonic"; cfg.terms.only(configurational="ideal").  contributions.show() lists the
# models of every term, the files they need and which of them are experimental.
cfg.terms.vibrational = None     # None = as prepared | "off" | "ZPE" | "harmonic" | "quasiharmonic" | "anharmonic"
cfg.hull.target_overlay = "anharmonic"  # anharmonic (MD) estimate reported next to the consistent hull ("off" to disable)
cfg.hull.temperatures = list(range(0, 1501, 50))   # T grid of the hull (quasi-harmonic: inside cfg.quasiharmonic.temperatures)
# operating T and the 0 K pruning window were fixed per phase in notebook 01 (plan.json)

# -- what the vibrational check reads and plots (None here = as prepared in notebook 01) ----------------------
# cfg.vibrational.sources = ["harmonic", "quasiharmonic", "anharmonic"]   # any subset; the most complete evidence decides
cfg.vibrational.save_figures = True   # phonons/<level>/ figures and stability/vibrational.png

TARGETS = None                           # None = every phase in database_root; or e.g. ["Cu3Au/L1_2"]

## Is everything there?
One table for the whole project: `done` is usable evidence; anything else is listed with the reason. Unfinished or failed
steps are also flagged `[incomplete]` in the verdict of every gate that reads them (`strict=True` makes such a gate `missing`).

In [ ]:
show_status()
display(status_table(status(), problems_only=True))

## Analyse the MD runs and assess the checks
`run_analysis` runs the calculator-free analysis steps (`mdstats`, `vdos`, `transition`) of every finished MD run; cached by manifest.

In [ ]:
run_analysis()
dirs = [os.path.join(cfg.phases.database_root, t) for t in (TARGETS or [])] or None
reports = {f"{r['material']}/{r['label']}": r for r in
           ([evaluate(d) for d in dirs] if dirs else evaluate_all())}
display(style_status(gate_table(reports)))
display(vibrational_table(reports))
for t, r in reports.items():
    display(show_stability_figure(r))
    if (r.get("vibrational") or {}).get("available"):
        print(t, "|", r["vibrational"]["message"])
        plot_vibrational(r); plt.show()          # bands (imaginary parts in red) / quasi-harmonic volumes / MD VDOS of cfg.vibrational.sources
    for f in r["flags"]:
        print(t, f)

## Hull in detail: consistent hull vs anharmonic overlay
The consistent curve compares like with like (every phase with the same method). The overlay adds
`F_VDOS - F_HA` of the target only: an *estimate* of the anharmonic stabilisation, not a hull.

In [ ]:
for t, r in reports.items():
    plot_free_energy_terms(r)
    plt.show()
    print("decomposition at operating T:", (r["thermodynamic"] or {}).get("decomposition"))

## A subset of checks / strict evidence
Only the listed checks are reported (the others show as `skipped`). `strict=True` turns a gate whose evidence is incomplete into `missing`.

In [ ]:
subset = {t: evaluate(os.path.join(cfg.phases.database_root, t), gates=["mechanical", "thermodynamic"],
                      save_figure=False, strict=True) for t in reports}
for t, r in subset.items():
    print(t, "| enabled:", r["enabled"], "| skipped:", r["skipped"], "| overall:", r["overall"], "| incomplete:", r["incomplete"])

## Perturbative anharmonic diagnostics (third-order force constants)
Only if notebook 01/02 planned the `perturbative` step (needs `phono3py`): kappa_RTA(T) and phonon linewidths. A diagnostic, never a gate and never a free-energy term (a perturbative free energy needs fc4 as well).

In [ ]:
for t, r in reports.items():
    if r.get("perturbative"):
        plot_perturbative(r); plt.show()
        print(t, r["perturbative"])
    else:
        print(t, "no perturbative result")

## Thresholds can change without recomputing

In [ ]:
with override(**{"gates.hull_ok_eV": 0.002, "gates.zero_frac_warn": 0.002, "gates.transition_rms_A": 0.2}):
    strict = {t: evaluate(os.path.join(cfg.phases.database_root, t), save_figure=False) for t in reports}
display(style_status(gate_table(strict)))

## Results from another code (DFT, another potential)
Notebook 03 only reads files. Put them in the folders notebook 01 created, in the layout of `heatup.core.layout`:

| evidence | file(s) | needed fields |
|---|---|---|
| relaxed structure + energy | `relaxation/CONTCAR`, `relaxation/energy.json` | `converged`, `energy_eV_per_atom`, `enthalpy_eV_per_atom`, `calculator` (a label: phases of one hull must share it) |
| mechanical | `elastic/elastic.json` | `C_GPa` (6x6, Voigt), `moduli` (`bulk_hill_GPa`, `shear_hill_GPa`, ...) |
| harmonic phonons (vibrational, thermodynamic) | `phonons/harmonic/harmonic.json` | `modes` (a `ModeSet` dict: `omega_meV`, `weight`), `imag_fraction`, `max_imag_meV`; `band` (`x`, `special_x`, `labels`, `kpoints`, `frequencies_meV`) to locate imaginary branches |
| MD (structural transition; vibrational via its VDOS, stored in `phonons/anharmonic/<T>K/`) | `md/<T>K/output.traj` (ASE, with momenta) + `md/<T>K/md.json` | `temperature_K`, `frame_dt_fs` (written last: marks the run complete) |

Files without a HeatUp manifest are reported `external` in the status table and accepted as they are.